In [3]:
!pip install pip torch sentencepiece maturin wheel setuptools datasets transformers

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.0-py3-none-any.whl (566 kB)
Using cached safetensors-0.6.2-cp38-abi3-macosx_11_0_arm64.whl (432 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.1.1
    Uninstalling huggingface_hub-1.1.1:
      Successfully uninstalled huggingface_hub-1.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers] [transformers]ub]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## code directory was creating naming problems for me 

In [4]:
# Turn the local "code/" folder into a runtime package named "code"
from pathlib import Path
import sys, types, importlib.machinery

code_dir = Path.cwd() / "code"
if not code_dir.is_dir():
    raise FileNotFoundError(f"Expected folder not found: {code_dir}")

# If something already occupied 'code' (e.g., stdlib), replace it for this session
if "code" in sys.modules:
    del sys.modules["code"]

pkg = types.ModuleType("code")
pkg.__path__ = [str(code_dir)]  # mark as a package
spec = importlib.machinery.ModuleSpec(name="code", loader=None, is_package=True)
spec.submodule_search_locations = [str(code_dir)]
pkg.__spec__ = spec
sys.modules["code"] = pkg

## import LanguageSpecificUnigramLMTokenizer

In [5]:
# Imports
from pathlib import Path
import pandas as pd
import sys

sys.path.append(str(Path.cwd() / "code"))

from language_specific_unigramlm import LanguageSpecificUnigramLMTokenizer

  from .autonotebook import tqdm as notebook_tqdm


## this was the smallest tokenizer I had ~9mb in size
## find base and language specific tokenizers

In [7]:
# Configuration / Paths
BASE_DIR = Path('example_tok/dslml_unimixlm_10k')
TOK_DIR = BASE_DIR / 'tokenizers'
BASE_JSON = BASE_DIR / 'base_tokenizer_10k.json'

LIMIT_LANGS = 500  # set to None to load all languages (may be slow)

assert BASE_JSON.exists(), f'Missing base tokenizer: {BASE_JSON}'
assert TOK_DIR.exists(), f'Missing tokenizers dir: {TOK_DIR}'
print('Base tokenizer:', BASE_JSON)
print('Tokenizers dir:', TOK_DIR)

Base tokenizer: example_tok/dslml_unimixlm_10k/base_tokenizer_10k.json
Tokenizers dir: example_tok/dslml_unimixlm_10k/tokenizers


## automatically finds all also you can limit number of tokenizers in upper cell

In [8]:
# Build {lang_code: tokenizer_path} mapping (expects files like langspec_soft_<code>.tokenizer.json)
lang_tok_paths = {
    p.stem.replace('langspec_soft_', '').replace('.tokenizer', ''): str(p)
    for p in sorted(TOK_DIR.glob('langspec_soft_*.tokenizer.json'))
}
assert len(lang_tok_paths) > 0, f'No language tokenizers found in {TOK_DIR}'
if LIMIT_LANGS:
    lang_tok_paths = dict(list(lang_tok_paths.items())[:int(LIMIT_LANGS)])
len(lang_tok_paths)

14

## loads normally

In [9]:
# Load unimixlm and per-language tokenizers (uses existing API)
lm = LanguageSpecificUnigramLMTokenizer()
lm.load(str(BASE_JSON), language_paths=lang_tok_paths)

2025-11-05 21:06:11,263 - code.tokenizer_formatting_utils - WARNING - example_tok/dslml_unimixlm_10k/base_tokenizer_10k.json looks byte-level but byte_fallback==False (currently known issue for HF Unigram Tokenizers).
2025-11-05 21:06:11,274 - code.standard_unigramlm - WARNING - No global probs for tokenizer at example_tok/dslml_unimixlm_10k/base_tokenizer_10k.json found
2025-11-05 21:06:11,339 - language_specific_unigramlm - INFO - Loaded per-lang tokenizer for EN-GB from example_tok/dslml_unimixlm_10k/tokenizers/langspec_soft_EN-GB.tokenizer.json
2025-11-05 21:06:11,350 - language_specific_unigramlm - INFO - Loaded per-lang tokenizer for EN-US from example_tok/dslml_unimixlm_10k/tokenizers/langspec_soft_EN-US.tokenizer.json
2025-11-05 21:06:11,362 - language_specific_unigramlm - INFO - Loaded per-lang tokenizer for ES-AR from example_tok/dslml_unimixlm_10k/tokenizers/langspec_soft_ES-AR.tokenizer.json
2025-11-05 21:06:11,373 - language_specific_unigramlm - INFO - Loaded per-lang toke

## checking pretokenizer and saving it, since we directly access rust pipeline it doesn't automatically use pretokenizer we need to specifically call it

In [10]:
pre_tok = lm._get_pretokenizer()
print('Pre-tokenizer:', pre_tok)

2025-11-05 21:06:24,119 - code.standard_unigramlm - INFO - Using pretokenizer from base tokenizer


Pre-tokenizer: ByteLevel(add_prefix_space=True, trim_offsets=True, use_regex=False)


In [11]:
pre_tok.pre_tokenize_str("This is a test.")[0][0]

'ĠThisĠisĠaĠtest.'

## just some examples sentences, dslml doesn't even have these languages

In [12]:
# Example sentences
sentences = [
    'Osogovo Dağları’nın güneybatı köşesinde bulunur. Coğrafya olarak bölge, maden bakımından zengindir. Romalılardan beri Probiştip’te madencilik yapılır.',
    'La région se situe dans le sud-ouest des monts Osogovo. Géographiquement, la région est riche en minerais. L\'extraction minière est pratiquée à Probištip depuis l\'époque romaine.',
    'The region is located in the southwestern part of the Osogovo Mountains. Geographically, the area is rich in minerals. Mining has been practiced in Probištip since Roman times.',
    'Регионът се намира в югозападната част на Осоговската планина. Географски районът е богат на минерали. Добивът на полезни изкопаеми се практикува в Пробищип от римско време.',
    'Область розташована на південному заході Осоговських гір. Географічно район багатий на корисні копалини. Видобуток корисних копалин практикується в Пробіштипі з часів Римської імперії.',
    'Regiunea este situată în partea de sud-vest a Munților Osogovo. Din punct de vedere geografic, zona este bogată în minerale. Exploatarea minieră este practicată în Probištip din vremea romană.',
    'Región se encuentra en la parte suroeste de las montañas Osogovo. Geográficamente, la zona es rica en minerales. La minería se practica en Probištip desde la época romana.',
    'Região está localizada na parte sudoeste das Montanhas Osogovo. Geograficamente, a área é rica em minerais. A mineração é praticada em Probištip desde os tempos romanos.',
    'Die Region liegt im Südwesten des Osogovo-Gebirges. Geographisch ist die Gegend reich an Mineralien. Der Bergbau wird in Probištip seit der Römerzeit betrieben.',
    'La regione si trova nella parte sud-occidentale dei Monti Osogovo. Geograficamente, la zona è ricca di minerali. L\'estrazione mineraria viene praticata a Probištip sin dai tempi dei Romani.',
    'Regionen ligger i den sydvästra delen av Osogovo-bergen. Geografiskt är området rikt på mineraler. Gruvdrift har bedrivits i Probištip sedan romartiden.',
    'Regiunea este situată în partea de sud-vest a Munților Osogovo. Din punct de vedere geografic, zona este bogată în minerale. Exploatarea minieră este practicată în Probištip din vremea romană.',
    'Región se encuentra en la parte suroeste de las montañas Osogovo. Geográficamente, la zona es rica en minerales. La minería se practica en Probištip desde la época romana.'
]
len(sentences)

13

## this loads language specific tokenizers to rust memory for fast acces.

In [13]:
# run ONCE after you load your per-language tokenizers
def prepare_rust_cache(lm):
    # pick a reference Unigram (first one)
    ref_lang, ref_info = next(iter(lm.per_lang_tok.items()))
    ref_tok = ref_info["tokenizer"]

    # check new Rust API
    need = ("set_weight_sets", "best_of_cached_weight_sets")
    if not all(hasattr(ref_tok.model, n) for n in need):
        raise RuntimeError("Unigram build lacks cached-weights API")

    ref_vocab = ref_tok.get_vocab()  # token -> id (stable order)
    V = len(ref_vocab)

    langs, weight_sets = [], []
    very_neg = -1e30  # floor for unseen tokens

    # align every language's weights to reference vocab order
    for lc, inf in lm.per_lang_tok.items():
        scores = inf["scores"]  # {token -> logp}
        w = [very_neg] * V
        for tok, rid in ref_vocab.items():
            lp = scores.get(tok)
            if lp is not None:
                w[rid] = lp
        langs.append(lc)
        weight_sets.append(w)

    # push all weights into Rust once (converted to f32 internally)
    ref_tok.model.set_weight_sets(weight_sets)

    # keep only minimal Python-side state
    lm._ref_tok_for_ws = ref_tok   # reference Unigram tokenizer
    lm._langs = langs              # language order used by the cache

prepare_rust_cache(lm)

## this is how you call it "pre_tok.pre_tokenize_str(sentences[i])[0][0]" we first need to push it through pretokenizer

In [14]:
# timing in milliseconds using cached Rust weights
import time

i = 0
t0 = time.perf_counter()

idx, toks, score = lm._ref_tok_for_ws.model.best_of_cached_weight_sets(pre_tok.pre_tokenize_str(sentences[i])[0][0])
best_lang = lm._langs[idx]

elapsed_ms = (time.perf_counter() - t0) * 1000.0
print(f"Single prediction time: {elapsed_ms:.3f} ms")

df_single = pd.DataFrame([{
    'sentence': sentences[i],
    'best_lang': best_lang,
    'tokens': ' '.join(toks),
}])
df_single

Single prediction time: 0.238 ms


,sentence,best_lang,tokens
0,Osogovo Dağları’nın güneybatı köşesinde bulunu...,sr,ĠO s og ovoĠ D a Ä Ł la r Ä ± âĢĻ n Ä ± n Ġ g ...


## just for comparison

In [15]:
# add timing in terms of miliseconds

import time

i=0
start_time = time.time()

# Predict with best_language_encode (single)
best_lang, toks = lm.best_language_encode(sentences[i])

end_time = time.time()
print(f"Single prediction time: {(end_time - start_time) * 1000:.4f} milliseconds")
rows = []
rows.append({
    'sentence': sentences[i],
    'best_lang': best_lang,
    'tokens': ' '.join(toks),
    })
df_single = pd.DataFrame(rows)
df_single

Single prediction time: 4.6291 milliseconds


,sentence,best_lang,tokens
0,Osogovo Dağları’nın güneybatı köşesinde bulunu...,sr,ĠO s og ovoĠ D a Ä Ł la r Ä ± âĢĻ n Ä ± n Ġ g ...
